In [3]:
import pandas as pd
import numpy as np
import altair as alt

pd.set_option("display.max_columns", None)

drivers_df = pd.read_csv("../../data/ergast/drivers.csv")
lap_times_df = pd.read_csv("../../data/ergast/lap_times.csv")
pit_stops_df = pd.read_csv("../../data/ergast/pit_stops.csv")
races_df = pd.read_csv("../../data/ergast/races.csv")
results_df = pd.read_csv("../../data/ergast/results.csv")
constructors_df = pd.read_csv("../../data/ergast/constructors.csv")


In [4]:
pit_stops_df.columns

Index(['raceId', 'driverId', 'stop', 'lap', 'time', 'duration',
       'milliseconds'],
      dtype='object')

## Lap and Pit Stop Times Dataframe

In [5]:
# Limit drivers_df to only required data
driver_columns_to_keep = ["driverId", "forename", "surname", "nationality"]
drivers_df = drivers_df[driver_columns_to_keep]


# Remove url from races_df
races_df = races_df.drop("url", axis = 1)

# Keep only races 2009 and on
races_df = races_df[races_df["year"] >= 2009]

# Limit races_df to only required data
race_name_columns_to_keep = ["raceId", "circuitId", "name", "year", "time"]
race_names_df = races_df[race_name_columns_to_keep]

# Rename columns in races_names_df to clear up the dataset
race_names_df = race_names_df.rename(columns = {
    "name": "circuitName",
    "time": "timeOfRace"
})



# Rename columns in pit_stop_df to clear up the dataset
pit_stops_df = pit_stops_df.rename(columns = {
    "stop": "pitStopNumber",
    "time": "pitStopTimeOfDay",
    "duration": "pitStopDuration",
    "milliseconds": "pitStopMilliseconds"
})



# Merge lap_times_df with drivers_df, race_names_df, and pit_stop_df
lap_times_df = pd.merge(lap_times_df, drivers_df, how = "left", on = "driverId")
lap_times_df = pd.merge(lap_times_df, race_names_df, how = "left", on = "raceId")
lap_times_df = pd.merge(lap_times_df, pit_stops_df, how = "left", on = ["raceId", "driverId", "lap"])



# Rename columns in lap_times_df to clear up the dataset
lap_and_pit_times_df = lap_times_df.rename(columns = {
    "milliseconds": "lapMilliseconds"
})

# Add boolean column for whether the lap had a pitstop or not
lap_and_pit_times_df["pitstop"] = ~lap_and_pit_times_df["pitStopNumber"].isnull()

# Drop all NA year values
lap_and_pit_times_df = lap_and_pit_times_df.dropna(subset = ["year"])

# Clean up constructor's file
constructor_columns_to_keep = ["constructorId", "name"]
constructors_df = constructors_df[constructor_columns_to_keep]
constructors_df = constructors_df.rename(columns = {"name": "constructorName"})

# Merge constructors_df with results_df
results_df = pd.merge(results_df, constructors_df, how = "left", on = "constructorId")

# Merge results_df with lap_and_pit_times_df
lap_and_pit_times_df = pd.merge(lap_and_pit_times_df, results_df[["raceId", "driverId", "constructorId", "constructorName"]], how = "left", on = ["raceId", "driverId"])

# Sort df by raceId, then driverId, then lap number
lap_and_pit_times_df = lap_and_pit_times_df.sort_values(by = ["raceId", "driverId", "lap"], ascending = [True, True, True])


# Preview processed df
pitstop_test_df = lap_and_pit_times_df[lap_and_pit_times_df["pitstop"] == True]
pitstop_test_df.head()
#lap_and_pit_times_df.head()

,raceId,driverId,lap,position,time,lapMilliseconds,forename,surname,nationality,circuitId,circuitName,year,timeOfRace,pitStopNumber,pitStopTimeOfDay,pitStopDuration,pitStopMilliseconds,pitstop,constructorId,constructorName
73,841,1,16,1,1:52.039,112039,Lewis,Hamilton,British,1.0,Australian Grand Prix,2011.0,06:00:00,1.0,17:28:24,23.227,23227.0,True,1,McLaren
93,841,1,36,2,1:53.298,113298,Lewis,Hamilton,British,1.0,Australian Grand Prix,2011.0,06:00:00,2.0,17:59:29,23.199,23199.0,True,1,McLaren
671,841,2,15,16,1:54.318,114318,Nick,Heidfeld,German,1.0,Australian Grand Prix,2011.0,06:00:00,1.0,17:27:41,22.994,22994.0,True,4,Renault
686,841,2,30,14,1:54.826,114826,Nick,Heidfeld,German,1.0,Australian Grand Prix,2011.0,06:00:00,2.0,17:51:32,25.098,25098.0,True,4,Renault
363,841,3,16,8,1:54.239,114239,Nico,Rosberg,German,1.0,Australian Grand Prix,2011.0,06:00:00,1.0,17:29:00,23.716,23716.0,True,131,Mercedes


In [ ]:
# Write the DataFrame to a CSV file at the specified location
#lap_and_pit_times_df.to_csv("../../data/lap_and_pit_times.csv", index = False)